In [ ]:
!pip install sentence_transformers --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: sentence_transformers
    Found existing installation: sentence-transformers 5.1.2
    Uninstalling sentence-transformers-5.1.2:
      Successfully uninstalled sentence-transformers-5.1.2


In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 137.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import logging
import matplotlib.pyplot as plt
from sentence_transformers import InputExample, CrossEncoder, LoggingHandler
from torch.utils.data import DataLoader
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
import torch
import torch.nn.functional as F
import numpy as np
import os
from datetime import datetime

import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
import json
from sentence_transformers import InputExample

def load_finder_triplets(path):
    examples = []
    with open(path, "r") as f:
        for line in f:
            item = json.loads(line)

            query = item["query"]
            pos = item["positive"]["text"]

            examples.append(InputExample(texts=[query, pos], label=1))

            for neg in item["negatives"]:
                examples.append(InputExample(
                    texts=[query, neg["text"]],
                    label=0
                ))
    return examples

In [ ]:
data_path = "finder_triplets_optimized.jsonl"
examples = load_finder_triplets(data_path)
print(f"Loaded {len(examples)} examples")

queries = [ex.texts[0] for ex in examples]
unique_queries = list(set(queries))
print(f"Total unique queries: {len(unique_queries)}")


splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_queries_idx, test_queries_idx = next(splitter.split(unique_queries, groups=unique_queries))

train_queries = [unique_queries[i] for i in train_queries_idx]
test_queries = [unique_queries[i] for i in test_queries_idx]

train_samples = [ex for ex in examples if ex.texts[0] in train_queries]
test_samples = [ex for ex in examples if ex.texts[0] in test_queries]

Loaded 13756 examples
Total unique queries: 3439


In [ ]:
from sentence_transformers import SentenceTransformer, losses

In [ ]:
import json
from sklearn.model_selection import GroupShuffleSplit
from sentence_transformers import InputExample

def load_finder_triplets(path):
    triplets = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            triplets.append({
                "query": item["query"],
                "positive": item["positive"]["text"],
                "negatives": [n["text"] for n in item["negatives"]]
            })
    return triplets

all_triplets = load_finder_triplets("finder_triplets_optimized.jsonl")

queries = [t["query"] for t in all_triplets]

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(all_triplets, groups=queries))

train_triplets = [all_triplets[i] for i in train_idx]
val_triplets   = [all_triplets[i] for i in val_idx]


In [ ]:
def triplets_to_pairs(triplets):
    return [
        InputExample(texts=[t["query"], t["positive"]])
        for t in triplets
    ]

train_examples = triplets_to_pairs(train_triplets)


In [ ]:
import torch
import numpy as np
from sentence_transformers.util import cos_sim

def evaluate_ranking_metrics(model, triplets, ks=[5, 10, 20]):
    metrics = {f"Recall@{k}": 0 for k in ks}
    metrics.update({f"Precision@{k}": 0 for k in ks})
    metrics.update({f"nDCG@{k}": 0 for k in ks})
    metrics["MRR"] = 0

    for t in triplets:
        query = t["query"]
        candidates = [t["positive"]] + t["negatives"]

        q_emb = model.encode(
            query, convert_to_tensor=True, normalize_embeddings=True
        )
        d_emb = model.encode(
            candidates, convert_to_tensor=True, normalize_embeddings=True
        )

        sims = cos_sim(q_emb, d_emb)[0]
        ranking = torch.argsort(sims, descending=True).cpu().tolist()

        # rank of the positive (index 0)
        rank = ranking.index(0) + 1
        metrics["MRR"] += 1 / rank

        for k in ks:
            topk = ranking[:k]

            # Recall
            if 0 in topk:
                metrics[f"Recall@{k}"] += 1

            # Precision
            metrics[f"Precision@{k}"] += int(0 in topk) / k

            # nDCG
            if 0 in topk:
                metrics[f"nDCG@{k}"] += 1 / np.log2(rank + 1)

    n = len(triplets)
    for key in metrics:
        metrics[key] /= n

    return metrics


In [ ]:
from sentence_transformers import SentenceTransformer, losses
best_mrr = 0.0

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
model.max_seq_length = 256

train_loader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16,   # increase if GPU allows
    drop_last=True
)

train_loss = losses.MultipleNegativesRankingLoss(model)

for epoch in range(1, 6):
    print(f"\nEpoch {epoch}/5")

    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=1,
        warmup_steps=100,
        optimizer_params={"lr": 2e-5},
        use_amp=True,
        show_progress_bar=True
    )

    val_metrics = evaluate_ranking_metrics(model, val_triplets)

    print("Validation metrics:")
    for k, v in val_metrics.items():
        print(f"  {k}: {v:.4f}")

    # Save best checkpoint
    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        model.save("models/finder_dense_encoder_best")
        print("✅ Saved new best model")


Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.



Epoch 1/5


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Validation metrics:
  Recall@5: 1.0000
  Recall@10: 1.0000
  Recall@20: 1.0000
  Precision@5: 0.2000
  Precision@10: 0.1000
  Precision@20: 0.0500
  nDCG@5: 0.6917
  nDCG@10: 0.6917
  nDCG@20: 0.6917
  MRR: 0.5891


...

Validation metrics:
  Recall@5: 1.0000
  Recall@10: 1.0000
  Recall@20: 1.0000
  Precision@5: 0.2000
  Precision@10: 0.1000
  Precision@20: 0.0500
  nDCG@5: 0.7033
  nDCG@10: 0.7033
  nDCG@20: 0.7033
  MRR: 0.6044


Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


✅ Saved new best model

Epoch 5/5


Step,Training Loss


Validation metrics:
  Recall@5: 1.0000
  Recall@10: 1.0000
  Recall@20: 1.0000
  Precision@5: 0.2000
  Precision@10: 0.1000
  Precision@20: 0.0500
  nDCG@5: 0.7078
  nDCG@10: 0.7078
  nDCG@20: 0.7078
  MRR: 0.6103
✅ Saved new best model
